# HAP-E — GoEmotions analysis — Mac / CPU path

Visualizes the precomputed sentence-level GoEmotions tags for the **HAP-E** corpus (`human` + 12 LLM authors across 6 genres), produced by [4a.GoEmotions_analysis_GPU.ipynb](4a.GoEmotions_analysis_GPU.ipynb) with [`cirimus/modernbert-large-go-emotions`](https://huggingface.co/cirimus/modernbert-large-go-emotions). **No model is loaded here** — every cell reads the per-document parquets in `data_processed/`. Safe to *Run All* on the Mac, provided those parquets are present.

**Primary contrast = `human` vs `machine`** (all 12 LLM groups pooled). Two artifacts:
- **Overview heatmap** — author × emotion across all 13 authors at once (raw per-doc mean on top; per-emotion z-score across authors on the bottom).
- **Figure-3 relative-usage dotplot** — machine-vs-human usage ratio per emotion on a log axis, paired on the shared `base_id` (the author-stripped parallel key, e.g. `acad_0001`). **Per-LLM** (12 panels) and **per-genre** variants are included **commented out** below.

Per-document metric defaults to the mean sentence probability (`goemotions_probs.parquet`); the dotplot independently loads per-label-threshold sentence counts (`goemotions_sentence_counts_threshold_perlabel.parquet`, from `4b.goemotions_perlabel_threshold_counts.py`). Requires `numpy`, `pandas`, `matplotlib`, `scipy`, `pyarrow`.

*Notebook scaffolding authored by Claude.*

## 1. HAP-E authors

The 13 author labels in display order (`human` first, then the 12 LLM groups), plus the `human` / `machine` pooling used by the primary contrast. `AUTHOR_SPECS` drives the heatmap's author ordering; authors absent from the loaded parquet are silently skipped.

In [ ]:
# HAP-E authors in display order (human first, then the 12 LLM groups).
AUTHOR_ORDER = [
    'human',
    'gpt-4o', 'gpt-4o-mini', 'gpt-5-mini',
    'llama-3-70b', 'llama-3-70b-instruct', 'llama-3-8b', 'llama-3-8b-instruct',
    'gemma-2-9b', 'gemma-2-9b-it', 'gemma-2-27b', 'gemma-2-27b-it',
    'claude-haiku-4-5',
]
LLM_AUTHORS = [a for a in AUTHOR_ORDER if a != 'human']

# AUTHOR_SPECS kept as an ordered dict (label -> metadata) so the heatmap orders authors
# human-first. group = 'human' | 'machine' is the pooling used by the primary contrast.
AUTHOR_SPECS = {
    a: dict(model=a, group=('human' if a == 'human' else 'machine'))
    for a in AUTHOR_ORDER
}

def author_group(a):
    """human stays human; every LLM pools into 'machine'."""
    return 'human' if a == 'human' else 'machine'

print(f'Defined {len(AUTHOR_SPECS)} HAP-E authors ({len(LLM_AUTHORS)} LLMs pooled as machine).')

## 2. Load precomputed per-document parquets (CPU, no model)

Rebuilds the variables every downstream cell uses (`df`, `emo_matrix`, `EMOTIONS`, `EMO_COLS`, `PROB_COLS`, `doc_sent_counts`) from one of the per-document parquets written by 4a. `df` also carries `genre`, `base_id`, and a pooled `author_group` (`human` / `machine`).

| `EMO_SOURCE` | file | `emo_*` value |
| --- | --- | --- |
| `'probs'` *(default)* | `goemotions_probs.parquet` | per-doc mean sentence sigmoid probability |
| `'frac'` | `goemotions_sentence_frac.parquet` | fraction of a doc's sentences carrying each emotion |
| `'counts'` | `goemotions_sentence_counts_threshold_perlabel.parquet` | per-label-threshold sentence counts (from 4b) |

*Authored by Claude.*

In [ ]:
# === Mac / CPU path: load precomputed per-document GoEmotions (no model, no CUDA) ===
# Rebuilds exactly the variables the heatmap / Setup / Figure-3 cells use
# (df, emo_matrix, EMOTIONS, EMO_COLS, PROB_COLS, doc_sent_counts) from a precomputed
# per-document parquet, plus genre / base_id / author_group on df.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
from IPython.display import display

# Which precomputed per-doc artifact to visualize:
#   'probs'  -> goemotions_probs.parquet           mean sentence sigmoid prob (default)
#   'frac'   -> goemotions_sentence_frac.parquet   fraction of sentences carrying each emotion
#   'counts' -> goemotions_sentence_counts_threshold_perlabel.parquet  per-label counts (from 4b)
# The Figure-3 dotplot loads per-label counts on its own (usage ratio), independent of EMO_SOURCE.
EMO_SOURCE = 'probs'
PERDOC_PATH, VALUE_LABEL = {
    'probs':  ('data_processed/goemotions_probs.parquet',           'mean sentence probability'),
    'frac':   ('data_processed/goemotions_sentence_frac.parquet',   'fraction of sentences'),
    'counts': ('data_processed/goemotions_sentence_counts_threshold_perlabel.parquet', 'sentences per emotion'),
}[EMO_SOURCE]
if not Path(PERDOC_PATH).exists():
    raise FileNotFoundError(
        f'{PERDOC_PATH} not found (cwd={Path.cwd()}). It is written by '
        f'4a.GoEmotions_analysis_GPU.ipynb -- run 4a on the HAP-E corpus first.')

perdoc   = pd.read_parquet(PERDOC_PATH)
EMOTIONS = [c[4:] for c in perdoc.columns if c.startswith('emo_')]
EMO_COLS  = [f'emo_{e}'  for e in EMOTIONS]
PROB_COLS = [f'prob_{e}' for e in EMOTIONS]

# Require the HAP-E per-doc schema. The pre-HAP-E LEAF parquets lack genre/base_id, which
# the Setup (genre) and Figure-3 (base_id pairing) cells need; fail with a clear pointer
# instead of a raw KeyError further down.
_missing = [c for c in ('doc_id', 'author', 'genre', 'base_id') if c not in perdoc.columns]
if _missing:
    raise RuntimeError(
        f'{PERDOC_PATH} is missing HAP-E column(s) {_missing}. This looks like a pre-HAP-E '
        f'cache (e.g. the old LEAF parquet). Regenerate the per-doc artifacts by running '
        f'4a.GoEmotions_analysis_GPU.ipynb on the HAP-E corpus (it writes doc_id/author/'
        f'genre/base_id + emo_<emotion> columns).')

# df = one row per (doc, author) with HAP-E metadata; _row indexes into emo_matrix.
_meta = [c for c in ['doc_id', 'author', 'genre', 'base_id', 'source_tag', 'n_sentences'] if c in perdoc.columns]
df = perdoc[_meta].reset_index(drop=True)
df['_row'] = np.arange(len(df))
df['author_group'] = np.where(df['author'] == 'human', 'human', 'machine')
emo_matrix = perdoc[EMO_COLS].to_numpy(dtype=np.float32)
doc_sent_counts = df['n_sentences'].to_numpy() if 'n_sentences' in df.columns else None

print(f'source={EMO_SOURCE}  {PERDOC_PATH}')
print(f'emo_matrix {emo_matrix.shape}  |  {len(EMOTIONS)} emotions  |  value = {VALUE_LABEL}')
print(df['author'].value_counts())
print('author_group:', df['author_group'].value_counts().to_dict())

### Overview heatmap

Compact author × emotion view across all 13 authors at once. Raw per-doc mean on top; the same matrix z-scored per emotion (column) on the bottom, so authors that systematically deviate on individual emotions stand out despite the raw scale being dominated by a few high-baseline labels.

*Authored by Claude.*

In [ ]:
# Author x emotion overview heatmaps (Mac path; needs AUTHOR_SPECS + the loader above).
# Top: raw per-doc mean. Bottom: per-emotion z-score across authors, which centers each
# column so small-but-systematic group differences surface (the raw scale is dominated by
# a few high-baseline emotions like neutral/admiration).
order = [a for a in AUTHOR_SPECS if a in set(df['author'])]
M = np.vstack([emo_matrix[df.loc[df['author'] == a, '_row'].values].mean(0) for a in order])
Z = (M - M.mean(0)) / (M.std(0) + 1e-9)

fig, axes = plt.subplots(2, 1, figsize=(16, 11))
panels = [
    (axes[0], M, 'viridis', VALUE_LABEL, (M.min(), M.max()),    f'GoEmotions {VALUE_LABEL} - author x emotion (raw)'),
    (axes[1], Z, 'coolwarm', 'z across authors', (-np.abs(Z).max(), np.abs(Z).max()), 'Same, per-emotion z-score across authors'),
]
for ax, mat, cmap, lbl, (vmin, vmax), ttl in panels:
    im = ax.imshow(mat, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(EMOTIONS))); ax.set_xticklabels(EMOTIONS, rotation=45, ha='right')
    ax.set_yticks(np.arange(len(order)));    ax.set_yticklabels(order)
    ax.set_title(ttl)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label=lbl)
plt.tight_layout(); plt.show()

## 3. Group definitions & validation summary

The group definitions feeding the relative-usage dotplot below. The **primary** setup is `human` vs `machine` (all 12 LLMs pooled); a per-genre breakdown follows. For each, prints group sizes (confirms wiring) and a **truncated** Welch summary — the top emotions by |mean diff| vs the `human` baseline on per-document mean sentence probabilities — just enough to sanity-check the direction/magnitude of the dotplot ratios. A **per-LLM** setup (each model separately) is included commented out.

*Authored by Claude.*

In [ ]:
import time
_t0 = time.time()
# === HAP-E setups: group definitions + validation summaries (no charts) ==============
# Per setup, print group sizes (confirms wiring) and a TRUNCATED Welch summary: the top
# emotions by |mean diff| vs the human baseline, on per-doc mean sentence probabilities.
# (The dotplots below use a different metric -- per-label threshold sentence fractions --
# so these numbers preview the qualitative signal, not the exact ratios.)
TOP_N_VALIDATE = 6   # emotions shown per non-baseline group (largest |mean diff|)

def _assign_groups(group_authors, genre=None):
    a2g = {a: g for g, authors in group_authors.items() for a in authors}
    sub = df.loc[df['author'].isin(a2g)].copy()
    if genre is not None:
        sub = sub.loc[sub['genre'] == genre]
    sub['group'] = sub['author'].map(a2g)
    return sub

def _validate_setup(title, group_authors, baseline_group, genre=None, top_n=TOP_N_VALIDATE):
    sub   = _assign_groups(group_authors, genre=genre)
    order = list(group_authors)
    sizes = sub['group'].value_counts().reindex(order)
    print(f'=== {title}  (baseline = {baseline_group}) ===')
    print('  group sizes:', {g: int(n) for g, n in sizes.items() if n == n})
    base = emo_matrix[sub.loc[sub['group'] == baseline_group, '_row'].values]
    rows = []
    for g in order:
        if g == baseline_group:
            continue
        gv = emo_matrix[sub.loc[sub['group'] == g, '_row'].values]
        if len(gv) == 0 or len(base) == 0:
            continue
        md = gv.mean(0) - base.mean(0)
        _, p = stats.ttest_ind(gv, base, axis=0, equal_var=False)
        for j in np.argsort(np.abs(md))[::-1][:top_n]:
            rows.append({'group': g, 'emotion': EMOTIONS[j],
                         'mean_diff': round(float(md[j]), 4),
                         'p': f'{p[j]:.1e}', 'sig': '*' if p[j] < 0.05 else ''})
    print(f'  top {top_n} emotions by |mean diff| vs baseline (Welch, per-doc mean prob):')
    print(pd.DataFrame(rows).to_string(index=False))
    print()

# Setup 1 (PRIMARY) -- human vs machine (all 12 LLMs pooled)
_validate_setup('Setup 1 - human vs machine (pooled)', {
    'human':   ['human'],
    'machine': LLM_AUTHORS,
}, 'human')

# Setup 2 -- human vs machine within each genre
for _g in sorted(df['genre'].dropna().unique()):
    _validate_setup(f'Setup 2 - human vs machine, genre={_g}', {
        'human':   ['human'],
        'machine': LLM_AUTHORS,
    }, 'human', genre=_g)

# Setup 3 (ALTERNATIVE, commented) -- human vs each LLM separately
# _validate_setup('Setup 3 - human vs each LLM', {
#     'human': ['human'],
#     **{a: [a] for a in LLM_AUTHORS},
# }, 'human')

print(f'Total runtime: {time.time()-_t0:6.2f}s')

## 4. Figure 3-style relative-usage dotplot

Adaptation of Fig. 3 in Reinhart et al., *Do LLMs write like humans?* (PNAS, [10.1073/pnas.2422455122](https://doi.org/10.1073/pnas.2422455122)), with GoEmotions labels substituted for Biber features.

**Per-document metric = per-label-threshold sentence counts** (`goemotions_sentence_counts_threshold_perlabel.parquet`, from `4b`): a sentence "fires" emotion *e* iff its sigmoid `prob_e ≥ THRESHOLDS[e]` (the model card's tuned per-label cuts). Multi-label — a sentence can fire zero, one, or several emotions.

**X-axis = usage relative to the `human` baseline** = group mean / baseline mean (`1×` = parity), log axis. Counts are **length-normalized** to the fraction of a doc's sentences firing the emotion (`normalize=True`), removing the LLM-vs-human length gap. `embarrassment` is excluded (noisy GoEmotions label).

**Pairing is on `base_id`** (the author-stripped parallel key, e.g. `acad_0001`): each panel is joined to the `human` baseline on `base_id`, and for the pooled **`machine`** panel each `base_id`'s value is the mean across the available LLM authors. Tests are paired Wilcoxon signed-rank with Bonferroni correction across the displayed emotion×panel grid. Effect size = pooled-SD Cohen *d_av* (`effect='d_av'`), since human and machine are independently authored and only anchored by the same source sample.

Each marker: **shape** = significance (triangle = Bonferroni p < 0.05, circle = n.s.), **fill** = effect size (filled = |Cohen *d*| ≥ 0.5), **colour** = GoEmotions valence (green = positive, red = negative, grey = ambiguous/neutral). The y-axis is grouped by valence; within each group emotions keep the |*d*| ranking.

The **primary** figure is `human` vs `machine` (pooled, single panel). **Per-LLM** (12 panels) and **per-genre** variants are included commented out.

*Authored by Claude.*

In [ ]:
# === (Optional) regenerate flat-0.30 baseline sentence counts on the Mac ============
# Normally written by 4a; recompute here CPU-only if the file is missing. A sentence fires
# emotion e iff its sigmoid prob_e >= EMO_THRESHOLD (flat 0.30, Google's published GoEmotions
# cut). Grouped by (doc_id, author, genre, base_id). NOTE: the dotplot below reads the
# PER-LABEL file from 4b, not this flat baseline.
from pathlib import Path as _Path
_SENT_PROBS_PATH    = 'data_processed/goemotions_sentence_probs.parquet'
_THRESH_COUNTS_PATH = 'data_processed/goemotions_sentence_counts_threshold.parquet'

# google-research/goemotions: `threshold` (calculate_metrics.py) and `eval_prob_threshold`
# (bert_classifier.py) both default to 0.3, applied globally -- no per-emotion cuts are published.
EMO_THRESHOLD = 0.30   # flat probability threshold for a sentence to "fire" an emotion

_sp = pd.read_parquet(_SENT_PROBS_PATH)
_fires = (_sp[[f'prob_{e}' for e in EMOTIONS]].to_numpy() >= EMO_THRESHOLD).astype(np.int64)
_fd = pd.DataFrame(_fires, columns=[f'emo_{e}' for e in EMOTIONS])
_keys = [k for k in ('doc_id', 'author', 'genre', 'base_id', 'source_tag') if k in _sp.columns]
_fd[_keys] = _sp[_keys].values
thresh_counts = (_fd.groupby(_keys, as_index=False)
                    [[f'emo_{e}' for e in EMOTIONS]].sum())
thresh_counts.to_parquet(_THRESH_COUNTS_PATH, index=False)

_M = thresh_counts[[f'emo_{e}' for e in EMOTIONS]].to_numpy()
print(f'Wrote {_THRESH_COUNTS_PATH}: {thresh_counts.shape} (flat threshold {EMO_THRESHOLD})')
print(f'mean emotions fired/doc {_M.sum(1).mean():.2f}  |  '
      f'sentences firing nothing {(_fires.sum(1) == 0).mean():.1%}  |  '
      f'firing >=2 {(_fires.sum(1) >= 2).mean():.1%}')
print(f'per-emotion mean count/doc range {_M.mean(0).min():.4f}-{_M.mean(0).max():.3f}')
del _sp, _fires, _fd

In [ ]:
# === Figure-3 relative-usage dotplot (sentence-count based) ========================
# x = usage relative to the human baseline = group mean count / baseline mean count
# (1 = parity), log axis. Per-doc metric = PER-LABEL threshold sentence counts
# (goemotions_sentence_counts_threshold_perlabel.parquet, from 4b), aligned to df by
# (doc_id, author); independent of EMO_SOURCE.
#
# PAIRING IS ON base_id (the author-stripped parallel key, e.g. 'acad_0001'): each panel is
# joined to the baseline on base_id. The pooled 'machine' panel averages, per base_id, across
# the available LLM authors. A panel label may carry an '@<genre>' suffix to restrict to one
# genre (used by the commented per-genre variant below).
#
# Each marker: shape ^ = paired Wilcoxon (Bonferroni) p < alpha, o = n.s.; fill = |Cohen d| >=
# dz_fill; color = emotion valence (green positive / red negative / gray ambiguous+neutral).
# effect='d_av' = pooled-SD Cohen d_av (human and machine are independently authored, only
# anchored by the same source sample, so the pairing carries little information).
# 'embarrassment' is dropped from the ranking/plot (noisy GoEmotions label).
# normalize=True -> per-doc metric is count / per-doc sentence count = length-invariant
# fraction of sentences (removes the LLM-vs-human length gap).
import math
from fractions import Fraction
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon

_SENT_PROBS_PATH = 'data_processed/goemotions_sentence_probs.parquet'
# Per-label-threshold counts from 4b.goemotions_perlabel_threshold_counts.py.
_PERLABEL_COUNTS_PATH = 'data_processed/goemotions_sentence_counts_threshold_perlabel.parquet'
_um = (df[['doc_id', 'author', '_row']]
       .merge(pd.read_parquet(_PERLABEL_COUNTS_PATH)[['doc_id', 'author'] + [f'emo_{e}' for e in EMOTIONS]],
              on=['doc_id', 'author'], how='left')
       .sort_values('_row'))
assert len(_um) == len(df) and _um[[f'emo_{e}' for e in EMOTIONS]].notna().all().all(), \
    'usage counts did not align 1:1 with df rows'
usage_matrix = _um[[f'emo_{e}' for e in EMOTIONS]].to_numpy(dtype=np.float64)

# Per-doc sentence count, aligned to df rows, for length normalization.
_ns = (pd.read_parquet(_SENT_PROBS_PATH, columns=['doc_id', 'author'])
         .groupby(['doc_id', 'author']).size().rename('n_sent').reset_index())
_nm = (df[['doc_id', 'author', '_row']]
       .merge(_ns, on=['doc_id', 'author'], how='left').sort_values('_row'))
sent_counts = _nm['n_sent'].to_numpy(dtype=np.float64)
with np.errstate(divide='ignore', invalid='ignore'):
    frac_matrix = np.where(sent_counts[:, None] > 0, usage_matrix / sent_counts[:, None], 0.0)
print(f'usage_matrix {usage_matrix.shape}  counts mean/doc range '
      f'{usage_matrix.mean(0).min():.3f}-{usage_matrix.mean(0).max():.3f}  |  '
      f'frac mean range {frac_matrix.mean(0).min():.4f}-{frac_matrix.mean(0).max():.3f}')

# eps pseudocounts guard division by near-zero baselines. Two scales: counts run ~0.001-20
# (EMO_EPS=1e-3); fractions are ~n_sent-times smaller (EMO_EPS_FRAC=1e-4).
EMO_EPS, EMO_EPS_FRAC, DZ_FILL = 1e-3, 1e-4, 0.5
DROP_EMOTIONS = ('embarrassment',)  # excluded from the dotplots (noisy GoEmotions label)

# Valence taxonomy = GoEmotions official sentiment grouping (Demszky et al. 2020).
# Neutral is folded into ambiguous (no separate 4th color).
EMO_POSITIVE = {'admiration', 'amusement', 'approval', 'caring', 'desire', 'excitement',
                'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief'}
EMO_NEGATIVE = {'anger', 'annoyance', 'disappointment', 'disapproval', 'disgust',
                'embarrassment', 'fear', 'grief', 'nervousness', 'remorse', 'sadness'}
# ambiguous (+ neutral): confusion, curiosity, realization, surprise, neutral
VALENCE_COLOR = {'positive': '#2ca02c', 'negative': '#d62728', 'ambiguous': '#7f7f7f'}


def _emo_valence(emo):
    if emo in EMO_POSITIVE:
        return 'positive'
    if emo in EMO_NEGATIVE:
        return 'negative'
    return 'ambiguous'   # confusion/curiosity/realization/surprise + neutral


def _nice_ratio_ticks(x_lo, x_hi, max_ticks=9):
    # Reinhart et al. PNAS Fig. 3 style: small integers above parity (1, 2, 3, 4) and unit
    # fractions below (1/2, 1/3, 1/4), log-symmetric about 1 (always kept). Spans too wide to
    # label that finely thin to powers of two, still rendered as integers / unit fractions.
    base = [1 / 4, 1 / 3, 1 / 2, 1.0, 2.0, 3.0, 4.0]
    cand = {v for v in base if x_lo <= v <= x_hi} | {1.0}
    k = 1
    while 2.0 ** -k >= x_lo or 2.0 ** k <= x_hi:
        for v in (2.0 ** -k, 2.0 ** k):
            if x_lo <= v <= x_hi:
                cand.add(v)
        k += 1
    cand = sorted(cand)
    if len(cand) <= max_ticks:
        return cand
    k_lo, k_hi = math.floor(math.log2(x_lo)), math.ceil(math.log2(x_hi))   # too wide: powers of 2
    return sorted(2.0 ** k for k in range(k_lo, k_hi + 1) if x_lo <= 2.0 ** k <= x_hi)


def _ratio_tick_label(v):
    # Reinhart PNAS Fig. 3 style: plain numbers at/above parity (1, 2, 3), unit fractions
    # below (1/2, 1/3, 1/4). No multiplier suffix.
    if v >= 1.0:
        return f'{v:g}'
    fr = Fraction(v).limit_denominator(64)
    return f'{fr.numerator}/{fr.denominator}'


def _emo_bonferroni(p):
    # Bonferroni-adjusted p-values: raw p times the number of tests, capped at 1.
    p = np.asarray(p, dtype=float)
    return np.clip(p * p.size, 0.0, 1.0)


def _label_rows(label):
    """Row mask for a panel/baseline label: a real author, 'machine' (all LLMs pooled), or
    '<label>@<genre>' to additionally restrict to one genre."""
    grp, genre = (label.split('@', 1) + [None])[:2] if '@' in label else (label, None)
    mask = (df['author'] != 'human') if grp == 'machine' else (df['author'] == grp)
    if genre is not None:
        mask = mask & (df['genre'] == genre)
    return mask


def _label_frame(label, normalize=False):
    """base_id-indexed emotion matrix for a label; pooled labels are averaged per base_id."""
    M = frac_matrix if normalize else usage_matrix
    rows = df.index[_label_rows(label)]
    vals = pd.DataFrame(M[df.loc[rows, '_row'].values],
                        index=df.loc[rows, 'base_id'].values)
    return vals.groupby(level=0).mean()


def _author_vals(label, normalize=False):
    return _label_frame(label, normalize).to_numpy()


def _paired_emo(baseline, panel, normalize=False):
    # Align baseline and panel on base_id (the shared parallel key). For pooled 'machine',
    # each base_id's value is the mean across the available LLM authors.
    b = _label_frame(baseline, normalize)
    g = _label_frame(panel, normalize)
    common = b.index.intersection(g.index)
    return b.loc[common].to_numpy(), g.loc[common].to_numpy()


def emo_shared_order_and_xlim(baseline, panels, top_k=15, eps=EMO_EPS, normalize=False, effect='dz'):
    # Rank emotions by largest |Cohen d| (either direction) across all panels; report shared
    # log x-limits in RATIO space over the chosen emotions.
    base_mean = _author_vals(baseline, normalize).mean(0)
    R = np.vstack([(_author_vals(g, normalize).mean(0) + eps) / (base_mean + eps) for g in panels])
    dz = np.zeros((len(panels), len(EMOTIONS)))
    for pi, g in enumerate(panels):
        bvals, gvals = _paired_emo(baseline, g, normalize)
        diff = gvals - bvals
        if effect == 'd_av':
            sd = np.sqrt((bvals.var(axis=0, ddof=1) + gvals.var(axis=0, ddof=1)) / 2)
        else:
            sd = diff.std(axis=0, ddof=1)
        dz[pi] = np.where(sd > 0, diff.mean(axis=0) / sd, 0.0)
    score = np.abs(dz).max(axis=0)            # peak |d| across panels, per emotion
    cand = [i for i in range(len(EMOTIONS)) if EMOTIONS[i] not in DROP_EMOTIONS]
    order = sorted(cand, key=lambda i: score[i], reverse=True)[:top_k]
    feats = [EMOTIONS[i] for i in order]
    sub = R[:, order]
    return feats, float(sub.min()) / 1.4, float(sub.max()) * 1.4


def emo_fig3_dotplot(baseline, panels, title, top_k=15, eps=EMO_EPS, alpha=0.05,
                     dz_fill=0.5, normalize=False, strip=(), feature_order=None,
                     x_lo=None, x_hi=None, group_by_valence=True, effect='dz'):
    # x = usage relative to baseline = group mean / baseline mean (count, or sentence
    # fraction when normalize=True). 1x = parity.
    E = len(EMOTIONS)
    ratios = np.zeros((E, len(panels)))      # group mean / baseline mean (plotted x)
    pvals  = np.ones((E, len(panels)))       # raw paired-Wilcoxon p
    dzs    = np.zeros((E, len(panels)))      # Cohen d on per-base_id differences
    for ci, g in enumerate(panels):
        bvals, gvals = _paired_emo(baseline, g, normalize)
        ratios[:, ci] = (gvals.mean(0) + eps) / (bvals.mean(0) + eps)
        diff = gvals - bvals
        for ei in range(E):
            d  = diff[:, ei]
            d  = d[np.isfinite(d)]
            if effect == 'd_av':       # pooled-SD Cohen d_av (ignores pairing)
                sd = math.sqrt((bvals[:, ei].var(ddof=1) + gvals[:, ei].var(ddof=1)) / 2) \
                    if d.size > 1 else 0.0
            else:                      # paired Cohen d_z = mean(diff) / sd(diff)
                sd = d.std(ddof=1) if d.size > 1 else 0.0
            dzs[ei, ci] = d.mean() / sd if sd > 0 else 0.0
            if d.size == 0 or np.all(d == 0):
                pvals[ei, ci] = 1.0          # Wilcoxon is undefined on all-zero diffs
            else:
                pvals[ei, ci] = wilcoxon(d, zero_method='wilcox', alternative='two-sided').pvalue

    if feature_order is not None:
        # Fixed shared y-axis: plot exactly these emotions, in this order.
        idx_of = {e: i for i, e in enumerate(EMOTIONS)}
        top_idx = [idx_of[e] for e in feature_order]
        feats_top = list(feature_order)
    else:
        cand = [i for i in range(len(EMOTIONS)) if EMOTIONS[i] not in DROP_EMOTIONS]
        score = np.abs(dzs).max(axis=1)       # peak |d| across panels, per emotion
        top_idx = sorted(cand, key=lambda i: score[i], reverse=True)[:top_k]
        feats_top = [EMOTIONS[i] for i in top_idx]

    if group_by_valence:
        # Group the y-axis by valence -- positive, then negative, then ambiguous --
        # preserving the within-group order (|d| ranking, or the passed feature_order).
        _vrank = {'positive': 0, 'negative': 1, 'ambiguous': 2}
        _vo = sorted(range(len(top_idx)),
                     key=lambda k: _vrank[_emo_valence(EMOTIONS[top_idx[k]])])
        top_idx   = [top_idx[k]   for k in _vo]
        feats_top = [feats_top[k] for k in _vo]

    # Bonferroni across the displayed grid (plotted emotions x panels) = the markers shown.
    grid  = np.array([[pvals[ei, ci] for ci in range(len(panels))] for ei in top_idx])
    qgrid = _emo_bonferroni(grid.ravel()).reshape(grid.shape)

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(2.3 * n + 2.0, 0.4 * len(top_idx) + 1.4),
                             sharey=True, sharex=True)
    if n == 1:
        axes = [axes]
    y = np.arange(len(top_idx))[::-1]
    if x_lo is None or x_hi is None:
        sub = ratios[top_idx]
        x_lo, x_hi = float(sub.min()) / 1.4, float(sub.max()) * 1.4
    xticks = _nice_ratio_ticks(x_lo, x_hi)

    for ax, ci in zip(axes, range(n)):
        for yi, ei in enumerate(top_idx):
            r     = ratios[ei, ci]
            sig   = qgrid[yi, ci] < alpha                       # shape = significance
            big   = abs(dzs[ei, ci]) >= dz_fill                 # fill  = effect size
            color = VALENCE_COLOR[_emo_valence(EMOTIONS[ei])]   # color = valence
            ax.scatter(r, y[yi], marker='^' if sig else 'o', s=70,
                       facecolors=color if big else 'none',
                       edgecolors=color, linewidth=1.1)
        ax.axvline(1.0, color='black', linestyle='-', linewidth=1.5)   # baseline = 1x
        ax.set_xscale('log')                                           # log is visual only; ticks set explicitly below
        ax.set_xlim(x_lo, x_hi)
        ax.set_xticks(xticks)
        ax.set_xticks([], minor=True)
        ax.xaxis.set_major_formatter(FuncFormatter(
            lambda v, _: _ratio_tick_label(v)))
        ax.grid(axis='x', which='both', linestyle=':', linewidth=0.4, alpha=0.5)
        ptitle = panels[ci]
        for sfx in strip:
            ptitle = ptitle.replace(sfx, '')
        ax.set_title(ptitle, fontsize=9)
        ax.set_xlabel('Rate (1 = human)')
    axes[0].set_yticks(y)
    axes[0].set_yticklabels(feats_top, fontsize=8)
    metric = 'sentence fraction' if normalize else 'sentence count'
    eff_label = 'd_av' if effect == 'd_av' else 'dz'
    fig.suptitle(
        f'{title}'
        f' | Wilcoxon Bonferroni p<{alpha}, filled |{eff_label}|≥{dz_fill}', fontsize=10)
    # Color legend = valence taxonomy.
    legend_handles = [
        Line2D([0], [0], marker='o', linestyle='', markersize=8,
               markerfacecolor=VALENCE_COLOR[v], markeredgecolor=VALENCE_COLOR[v], label=lab)
        for v, lab in [('positive', 'positive'), ('negative', 'negative'),
                       ('ambiguous', 'ambiguous / neutral')]]
    fig.legend(handles=legend_handles, loc='lower center', ncol=3, fontsize=8,
               frameon=False, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()

    # Always report effect sizes: tidy ratio / d / Bonferroni-p table for the displayed grid.
    eff = pd.DataFrame([
        {'emotion': feats_top[yi], 'panel': panels[ci],
         'ratio': round(float(ratios[ei, ci]), 4), f'cohen_{eff_label}': round(float(dzs[ei, ci]), 4),
         'p_bonf': round(float(qgrid[yi, ci]), 4),
         'sig': '*' if qgrid[yi, ci] < alpha else ''}
        for ci in range(n) for yi, ei in enumerate(top_idx)
    ])
    print(f'Effect sizes - {title} ({metric}):')
    print(eff.to_string(index=False))

    # Return value: wide ratio frame (panel -> per-emotion ratio).
    return pd.DataFrame({g: ratios[top_idx, ci] for ci, g in enumerate(panels)}, index=feats_top)


### Figure 3 — machine vs human, and each LLM vs human

Baseline = `human`. Two figures run:
1. **Pooled** — a single `machine` panel (all 12 LLMs averaged per `base_id`).
2. **Per-LLM** — one panel per specific LLM (12 panels), so the models are distinguishable side by side; all panels share one feature order and x-limit.

Both use the length-normalized sentence-fraction metric and |Cohen *d_av*|, y-axis grouped by valence. A per-genre view (`machine@<genre>`) is included commented out.

In [ ]:
# === PRIMARY: human vs machine (all 12 LLMs pooled) ================================
SHARED, XLO, XHI = emo_shared_order_and_xlim(
    'human', ['machine'], eps=EMO_EPS_FRAC, normalize=True, effect='d_av')
print('Shared y-axis (top-15 most divergent emotions, machine vs human):')
print(SHARED)

fig3_machine = emo_fig3_dotplot(
    'human', ['machine'], 'GoEmotions — machine vs human (pooled)',
    eps=EMO_EPS_FRAC, normalize=True, effect='d_av',
    feature_order=SHARED, x_lo=XLO, x_hi=XHI,
)
print('Usage relative to human (1× = parity) — machine pooled:')
display(fig3_machine.round(3))

# === PER-LLM: one panel per specific LLM (each vs human) ==========================
# Distinguishes the individual models side by side; all panels share one feature order
# and x-limit so they read row-for-row. Restrict to LLM authors actually present in the
# loaded parquet: an absent author gives an empty panel, whose all-NaN ratios make the
# shared x-limits NaN and crash set_xlim. (Matches the heatmap's "absent authors skipped".)
LLM_PRESENT = [a for a in LLM_AUTHORS if a in set(df['author'])]
if len(LLM_PRESENT) < len(LLM_AUTHORS):
    print(f'\nskipping absent LLM(s): {sorted(set(LLM_AUTHORS) - set(LLM_PRESENT))}')

SHARED_LLM, LLM_XLO, LLM_XHI = emo_shared_order_and_xlim(
    'human', LLM_PRESENT, eps=EMO_EPS_FRAC, normalize=True, effect='d_av')
print(f'\nShared y-axis (top-15 most divergent across {len(LLM_PRESENT)} LLMs):')
print(SHARED_LLM)

fig3_per_llm = emo_fig3_dotplot(
    'human', LLM_PRESENT, 'GoEmotions — each LLM vs human',
    eps=EMO_EPS_FRAC, normalize=True, effect='d_av',
    feature_order=SHARED_LLM, x_lo=LLM_XLO, x_hi=LLM_XHI,
)
print('Usage relative to human (1× = parity) — per LLM:')
display(fig3_per_llm.round(3))

# === ALTERNATIVE (commented): machine vs human, one panel per genre ===============
# GENRE_PANELS = [f'machine@{g}' for g in sorted(df['genre'].dropna().unique())]
# SHARED_G, G_XLO, G_XHI = emo_shared_order_and_xlim(
#     'human', GENRE_PANELS, eps=EMO_EPS_FRAC, normalize=True, effect='d_av')
# fig3_per_genre = emo_fig3_dotplot(
#     'human', GENRE_PANELS, 'GoEmotions — machine vs human by genre',
#     eps=EMO_EPS_FRAC, normalize=True, effect='d_av', strip=('machine@',),
#     feature_order=SHARED_G, x_lo=G_XLO, x_hi=G_XHI,
# )
# display(fig3_per_genre.round(3))